# 질감 기준 필라멘트 게이트 확인 (전처리만)

모델 없이 **게이트만** 돌려서, 어떤 사진이 판정 보류되는지 확인하는 노트북입니다.
- 새 게이트: 고정 크롭 안에서 매끈하고 밝고 색이 옅은 영역(흰 출력물)의 면적 비율. 0.1 미만이면 보류
- 기존 게이트: Otsu 밝은 픽셀 비율 0.25 미만이면 보류 (원욱님·두현님 코드)
- 계산 방법과 코드에 넣는 법은 `README.md`와 `filament_gate.py` 맨 위 설명을 보세요.

**Drive `산공종설/`에 둘 파일**
- `underfill_originals_4000.zip` (원본, `normal/`, `underfill_50/`)
- `filament_gate.py` (이 묶음에 들어 있음)
- (선택) 두현님 `TalkFile_share_pipeline_3models_20261002.zip`: 있으면 6번에서 "매번 틀린 151장"을 게이트가 얼마나 거르는지 계산

GPU는 필요 없습니다. 전체 2~3분이면 끝나고, 결과는 Drive `산공종설/results/gate_check/`에 저장됩니다.

## 1. Drive 연결 + 파일 확인

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os, glob
DRIVE_DIR = "/content/drive/MyDrive/산공종설"   # 폴더 이름 다르면 여기만 바꾸기
OUT = f"{DRIVE_DIR}/results/gate_check"
os.makedirs(OUT, exist_ok=True)
DH_ZIP = f"{DRIVE_DIR}/TalkFile_share_pipeline_3models_20261002.zip"   # 선택
for f in ("underfill_originals_4000.zip", "filament_gate.py"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)
print("OK  " if os.path.exists(DH_ZIP) else "없음(선택)", os.path.basename(DH_ZIP))

## 2. 원본 풀기 + 게이트 모듈 불러오기
마지막 줄에 `4000`과 `self-check ok`가 나와야 합니다.

In [ ]:
import sys, shutil
if len(glob.glob("/content/orig/**/*.jpg", recursive=True)) < 4000:
    !cp "{DRIVE_DIR}/underfill_originals_4000.zip" /content/
    !unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
shutil.copy(f"{DRIVE_DIR}/filament_gate.py", "/content/filament_gate.py")
sys.path.insert(0, "/content")
import importlib, filament_gate as fg
importlib.reload(fg); fg._self_check()
print(len(glob.glob("/content/orig/**/*.jpg", recursive=True)), "장 | self-check ok | 기준", fg.THRESHOLD)

## 3. 4,000장 게이트 값 계산 (2분 안팎)
사진마다 새 비율(`new`)과 기존 Otsu 비율(`otsu`)을 계산해 `gate_values.csv`로 저장합니다. 이미 있으면 불러오기만 합니다.

In [ ]:
import cv2, time, numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

paths = sorted(glob.glob("/content/orig/normal/*.jpg")) + sorted(glob.glob("/content/orig/underfill_50/*.jpg"))
gv_file = f"{OUT}/gate_values.csv"
if os.path.exists(gv_file):
    gv = pd.read_csv(gv_file, encoding="utf-8-sig"); print("저장된 값 불러옴")
else:
    t0 = time.time()
    def one(p):
        crop = fg.crop_roi(cv2.imread(p))      # cv2라 BGR
        return fg.filament_ratio(crop, bgr=True), fg.otsu_ratio(crop, bgr=True)
    with ThreadPoolExecutor(4) as ex:
        vals = list(ex.map(one, paths))
    gv = pd.DataFrame({"source": [f"{Path(p).parent.name}/{Path(p).name}" for p in paths],
                       "label": [Path(p).parent.name for p in paths],
                       "frame": [int(Path(p).name[:8]) for p in paths],
                       "new": [round(v[0], 4) for v in vals], "otsu": [round(v[1], 4) for v in vals]})
    gv.to_csv(gv_file, index=False, encoding="utf-8-sig")
    print(f"완료: {time.time()-t0:.0f}초")
print(len(gv), "장")

## 4. 기존 vs 새 게이트: 몇 장, 어떤 사진이 겹치나
- 위 표: 게이트별 보류 장수 (클래스별)
- 아래 표: 두 게이트가 거른 사진의 겹침
- 그림: 두 값의 분포. 점선이 각 기준

In [ ]:
import matplotlib.pyplot as plt
OLD_T, NEW_T = 0.25, fg.THRESHOLD
gv["old_held"] = gv.otsu < OLD_T
gv["new_held"] = gv.new < NEW_T

t = pd.DataFrame({"기존(Otsu<0.25)": gv.groupby("label").old_held.sum(),
                  f"새(질감<{NEW_T})": gv.groupby("label").new_held.sum()})
t.loc["합계"] = t.sum(); t.loc["전체 대비"] = (t.loc["합계"] / len(gv) * 100).round(1).astype(str) + "%"
print(t, "\n")

ov = pd.Series({"둘 다 보류": (gv.old_held & gv.new_held).sum(),
                "기존만 보류": (gv.old_held & ~gv.new_held).sum(),
                "새 게이트만 보류": (~gv.old_held & gv.new_held).sum(),
                "둘 다 통과": (~gv.old_held & ~gv.new_held).sum()})
print(ov.to_string())

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
for lab, d in gv.groupby("label"):
    ax[0].hist(d.otsu, bins=60, alpha=.5, label=lab); ax[1].hist(d.new, bins=60, alpha=.5, label=lab)
ax[0].axvline(OLD_T, c="k", ls="--"); ax[1].axvline(NEW_T, c="r", ls="--")
ax[0].set_title("old gate: Otsu bright ratio"); ax[1].set_title("new gate: texture filament ratio")
for a in ax: a.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/gate_hist.png", dpi=100); plt.show()

## 5. 새 게이트가 보류한 사진: 10장마다 1장
보류된 사진을 클래스별 프레임 순서로 세운 뒤 10장마다 1장씩 보여 줍니다. **빨간색 = 게이트가 출력물로 판단한 영역**입니다. 빨간 영역이 화면의 10% 미만이라 보류된 것입니다.

확인할 것: 정말 판정할 정보가 없는 사진(베드만, 출력물이 가장자리에 살짝)인지, 출력물이 잘 보이는데 보류된 사진은 없는지.

그림은 `results/gate_check/held_every10_p01.png` 등으로 저장됩니다. 보류 목록 전체는 `held_new_gate.csv`에 있습니다.

In [ ]:
STEP = 10   # 몇 장마다 1장 보여 줄지

def show_grid(rows, title, fname, ncol=6):
    for pg in range(0, len(rows), ncol * 4):
        chunk = rows.iloc[pg:pg + ncol * 4]
        nrow = (len(chunk) + ncol - 1) // ncol
        fig, axs = plt.subplots(nrow, ncol, figsize=(2.6 * ncol, 2.9 * nrow), squeeze=False)
        for a in axs.flat: a.axis("off")
        for a, (_, r) in zip(axs.flat, chunk.iterrows()):
            crop = fg.crop_roi(cv2.imread(f"/content/orig/{r.source}"))
            a.imshow(fg.overlay(crop, bgr=True))
            a.set_title(f"{r.label[:9]} #{r.frame}\nnew={r.new:.2f} otsu={r.otsu:.2f}", fontsize=8)
        fig.suptitle(f"{title} ({pg+1}-{pg+len(chunk)} / {len(rows)})")
        plt.tight_layout(); plt.savefig(f"{OUT}/{fname}_p{pg // (ncol*4) + 1:02d}.png", dpi=80); plt.show(); plt.close(fig)

held = gv[gv.new_held].sort_values(["label", "frame"])
held.to_csv(f"{OUT}/held_new_gate.csv", index=False, encoding="utf-8-sig")
sample = pd.concat([d.iloc[::STEP] for _, d in held.groupby("label")])
print(f"새 게이트 보류 {len(held)}장 중 {len(sample)}장 표시 (클래스별 {STEP}장마다 1장)")
show_grid(sample, f"new gate held, every {STEP}th", "held_every10")

### 5-1. 기존 게이트만 보류한 사진 (새 게이트는 통과)
기존 게이트는 걸렀지만 새 게이트는 통과시킨 사진입니다. 출력물이 실제로 보이는 사진인지 확인합니다.

In [ ]:
old_only = gv[gv.old_held & ~gv.new_held].sort_values(["label", "frame"])
print(f"{len(old_only)}장")
if len(old_only):
    show_grid(old_only, "old gate only (new gate passed)", "old_only")

## 6. (선택) 두현님 "매번 틀린 151장"을 얼마나 거르나
두현님 zip의 `gate_check_per_frame.csv`(사진별로 9번 학습 중 몇 번 틀렸는지)를 써서, 두현님 보고서 4장 표와 같은 형식으로 계산합니다. **모델 없이 게이트만 평가**하는 방법입니다.
- `매번 틀린 151장 중 보류`: 많을수록 좋음
- `항상 맞힌 사진 중 보류`: 적을수록 좋음 (멀쩡한 사진을 버린 것)

두현님 파일은 기존 게이트에 걸린 49장을 뺀 3,951장 기준입니다. 새 기준 0.1은 이 파일을 보기 전에 겹쳐 그리기만 보고 정한 값입니다.

In [ ]:
import zipfile
dh_csv = None
if os.path.exists(DH_ZIP):
    with zipfile.ZipFile(DH_ZIP) as z:
        name = next(n for n in z.namelist() if n.endswith("results/gate_check_per_frame.csv"))
        dh_csv = z.extract(name, "/content/dh")
if dh_csv is None:
    print("두현님 zip이 없어서 건너뜀")
else:
    dh = pd.read_csv(dh_csv, encoding="utf-8-sig")
    dh["source"] = dh.source.str.replace("\\", "/", regex=False)
    d = dh.merge(gv[["source", "new", "otsu"]].rename(columns={"otsu": "otsu_mine"}), on="source", how="left")
    print("매칭 안 된 사진:", d.new.isna().sum(), "| Otsu 값 차이 최대:", round((d.otsu_ratio - d.otsu_mine).abs().max(), 4))
    bad, good = d.wrong_of_9 == 9, d.wrong_of_9 == 0
    rows = []
    for name, col, th in [("기존 Otsu<0.25 (현재)", "otsu", 0.25), ("기존 Otsu<0.35", "otsu", 0.35),
                          ("기존 Otsu<0.45", "otsu", 0.45), ("기존 Otsu<0.50", "otsu", 0.50),
                          ("새 질감<0.05", "new", 0.05), (f"새 질감<{NEW_T} (제안)", "new", NEW_T), ("새 질감<0.20", "new", 0.20)]:
        h_all = gv[col] < th
        h = d[col if col == "new" else "otsu_mine"] < th
        rows.append({"게이트": name,
                     f"매번 틀린 {bad.sum()}장 중 보류": f"{(h & bad).sum()}장 ({(h & bad).sum()/bad.sum():.0%})",
                     f"항상 맞힌 {good.sum()}장 중 보류": f"{(h & good).sum()}장 ({(h & good).sum()/good.sum():.0%})",
                     "4,000장 중 보류": f"{h_all.sum()}장 ({h_all.mean():.1%})"})
    res = pd.DataFrame(rows); res.to_csv(f"{OUT}/hard151_check.csv", index=False, encoding="utf-8-sig")
    print(res.to_string(index=False))